# Exercise 9: series

* pandas series vs numpy arrays [explanation](https://jakevdp.github.io/PythonDataScienceHandbook/03.01-introducing-pandas-objects.html)

### Common series operations
These are the most common series operations we use. Refer to the `pandas` docs for even more!

* Getting dates, hours, minutes from datetime types (`df.datetime_col.dt.date`)
* Parsing strings (`df.string_col.str.split('_')`)

### Common geoseries operations
These are the most common. Refer to the `geopandas` docs for even more!

* `distance` between 2 points or a point to a polygon or line [docs](https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoSeries.distance.html)
* `intersects`: [docs](https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoSeries.intersects.html)
* `within`: [docs](https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoSeries.within.html)
* `contains`: [docs](https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoSeries.contains.html)

In fact, we've often used geoseries methods without even realizing it. Often, we'd create a new column that stores either the line's length or a polygon's area. `gdf.geometry` is a geoseries, and we call methods on that geoseries, and add that as a new column.

For calculations like `length`, `area`, and `distance`, we need to use a projected CRS that has units like meters or feet. We cannot use decimal degrees (do not use WGS 84 / `EPSG:4326`)! Distance calculations must be done only once the spherical 3D Earth has been converted into a 2D plane.

* `length`: get the length of a line (`gdf.geometry.length`)
* `area`: get the area of a polygon (`gdf.geometry.area`)
* `centroid`: get the centroid of a polygon (`gdf.geometry.centroid`)
* `x`: get the x coordinate of a point (`gdf.geometry.x`)
* `y`: get the y coordinate of a point (`gdf.geometry.y`)

### Arrays
* Occasionally, we may even use arrays, especially when the datasets get even larger but we have simple mathematical calculations
* If we need to apply an exponential decay function to a distance column, we essentially want to multiple `distance` by some number
* Since this exponential decay function is somewhat custom and requires us to write our own formula, we would extract the column as a series (`df.distance`) and multiply each value by some other number.
* Even quicker is to use `numpy` with `distance_array = np.array(df.distance)` and get `exponential_array = distance_array*some_number`

In [1]:
import geopandas as gpd
import intake
import numpy as np
import pandas as pd

catalog = intake.open_catalog("shared_data_catalog.yml")

If you're asking how far is a transit stop from the interstate, you'd want the distance of every point (every row) compared to an interstate highway geometry.

Let's prep the datasets to use series / geoseries to do this.

In [2]:
stops = catalog.ca_transit_stops.read()[["agency", "stop_id", 
                                         "stop_name", "geometry"]]
highways = catalog.state_highway_network.read()

Since we want to know the distance from a stop's point to the interstate generally, we need a dissolve. We don't want to compare the distance against the I-5, the I-10 individually, but to the interstate system as a whole.

In [3]:
interstates = (highways[highways.RouteType=="Interstate"]
               .dissolve()
               .reset_index()
               [["geometry"]]
              )

In [4]:
# This is still a gdf, just with 1 column
type(interstates)

geopandas.geodataframe.GeoDataFrame

In [5]:
# Pulling out the individual column, it becomes a series/geoseries.
# It's a geoseries here because we had a gdf. 
# If it was a df, it would be a series.
print(type(stops.geometry))
print(type(interstates.geometry))

<class 'geopandas.geoseries.GeoSeries'>
<class 'geopandas.geoseries.GeoSeries'>


Distance is something you can calculate using `geopandas`.

Specifically, it takes a geoseries on the left, and either a geoseries or a single geometry on the right.

An example of having 2 geoseries would be comparing the distance between 2 points. On the left, it would be a geoseries of the origin points and on the right, destination points.

In [6]:
# We get a warning if we leave it in EPSG:4326!
stops.geometry.distance(interstates.geometry.iloc[0])

/tmp/ipykernel_763/2451250416.py:2: UserWarning: Geometry is in a geographic CRS. Results from 'distance' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  stops.geometry.distance(interstates.geometry.iloc[0])


0         0.045378
1         0.006873
2         0.006355
3         0.008839
4         0.045377
            ...   
131997    0.020412
131998    0.009226
131999    0.002394
132000    0.009510
132001    0.009668
Length: 132002, dtype: float64

In [7]:
stops_geom = stops.to_crs("EPSG:2229").geometry
interstates_geom = interstates.to_crs("EPSG:2229").geometry.iloc[0]

In [8]:
distance_series = stops_geom.distance(interstates_geom)

In [9]:
# Let's make sure that for every stop, a distance is calculated
print(f"# rows in stops: {len(stops_geom)}")
print(f"# rows in stops: {len(distance_series)}")

# rows in stops: 132002
# rows in stops: 132002


In [10]:
# distance is numeric, not a geometry, so we're back to being a series
type(distance_series)

pandas.core.series.Series

What can we do with this? 

We usually add it as a new column. Since we did nothing to shift the index, we can just attach the series back to our gdf.

Getting a distance calculation using geoseries is much quicker than a row-wise lambda function where you calculate the distance.

```
Alternative method that's slower:
      
interstate_geom = interstates.geometry.iloc[0]

stops = stops.assign(
   distance = stops.geometry.apply(
         lambda x: x.distance(interstate_geom))
)   
```

In [11]:
stops = stops.assign(
    distance_to_interstate = distance_series
)

In [12]:
#%%timeit
#distance_series = stops_geom.distance(interstates_geom)

In [13]:
#%%timeit
#stops.assign(
#   distance = stops.geometry.apply(
#         lambda x: x.distance(interstates_geom))
#)   

In [14]:
import dask_geopandas as dg

stops_gddf = dg.from_geopandas(stops, npartitions=2)
stops_geom_dg = stops_gddf.to_crs("EPSG:2229").geometry

In [15]:
#%%timeit

#distance_series = stops_geom_dg.distance(interstates_geom)

## To Do

* Use the `stop_times` table and `stops` table.
* Calculate the straight line distance between the first and last stop for each trip. Call this column `trip_distance`
* Calculate the distance between each stop to the nearest interstate. For each trip, keep the value for the stop that's the closest to the interstate. Call this column `shortest_distance_hwy`.
* For each trip, add these 2 new columns, but use series, geoseries, and/or arrays to assign it.
* Provide a preview of the resulting df (do not export)

In [16]:
GCS_FILE_PATH = ("gs://calitp-analytics-data/data-analyses/"
                 "rt_delay/compiled_cached_views/"
                )

analysis_date = "2023-01-18"
STOP_TIMES_FILE = f"{GCS_FILE_PATH}st_{analysis_date}.parquet"
STOPS_FILE = f"{GCS_FILE_PATH}stops_{analysis_date}.parquet"
highways = catalog.state_highway_network.read()

In [17]:
interstates = highways.loc[highways.RouteType == "Interstate"].to_crs("EPSG:3310")

interstates.head()

,Route,County,District,RouteType,Direction,geometry
42,5,COL,3,Interstate,NB,"MULTILINESTRING ((-178196.868 113423.347, -178..."
43,5,COL,3,Interstate,SB,"MULTILINESTRING ((-177795.613 112730.040, -177..."
44,5,FRE,6,Interstate,NB,"LINESTRING (-8832.137 -215819.170, -8879.531 -..."
45,5,FRE,6,Interstate,SB,"LINESTRING (-8854.507 -215842.986, -8879.302 -..."
46,5,GLE,3,Interstate,NB,"LINESTRING (-188733.365 154343.165, -188732.87..."


In [18]:
stop_times = pd.read_parquet(STOP_TIMES_FILE)
stop_times.tail()

,feed_key,trip_id,stop_id,stop_sequence,timepoint,arrival_sec,departure_sec,arrival_hour,departure_hour
3589926,fdf54bff79f705767670a13db867d3f7,1230080,831487,16,0.0,33387,33387,9,9
3589927,fdf54bff79f705767670a13db867d3f7,1008020,831808,40,0.0,62452,62452,17,17
3589928,fdf54bff79f705767670a13db867d3f7,646040,831998,21,0.0,34365,34365,9,9
3589929,fdf54bff79f705767670a13db867d3f7,1432040,831532,36,0.0,33503,33503,9,9
3589930,fdf54bff79f705767670a13db867d3f7,21080,831017,30,1.0,37140,37140,10,10


In [19]:
stops = gpd.read_parquet(STOPS_FILE).to_crs("EPSG:3310")
stops_duplicated = stops.loc[
    stops.set_index(["feed_key", "stop_id"]).index.duplicated()
]
# For some reason there is exactly one stop where feed_key and stop_id isn't a unique identifier, 
# which means we can't merge it with stop_times, so we'll just remove one of the duplicates
assert len(stops_duplicated) <= 1
stops = stops.drop(stops_duplicated.index)
stops.head()

,feed_key,stop_id,stop_key,stop_name,route_type_0,route_type_1,route_type_2,route_type_3,route_type_4,route_type_5,route_type_6,route_type_7,route_type_11,route_type_12,missing_route_type,geometry
0,6adf6cd9b6d24ab4ee8ee220e3697a73,15193,d4eb0920e7e256606df449c31b3c3e6a,Vanowen / Encino,NaN,NaN,NaN,69.0,NaN,NaN,None,None,None,None,None,POINT (136891.364 -423571.990)
1,6adf6cd9b6d24ab4ee8ee220e3697a73,14025,038cca58ef5f071ff5c94b8213989f87,Vermont / 110th,NaN,NaN,NaN,107.0,NaN,NaN,None,None,None,None,None,POINT (157907.732 -451883.320)
2,6adf6cd9b6d24ab4ee8ee220e3697a73,15638,06b1447efcc028791c8409d65fa3b3ee,3rd / Hobart,NaN,NaN,NaN,143.0,NaN,NaN,None,None,None,None,None,POINT (156427.875 -437101.095)
3,6adf6cd9b6d24ab4ee8ee220e3697a73,10244,87f19e30889f90d25e6dee49f04c4985,Vernon / Hooper,NaN,NaN,NaN,97.0,NaN,NaN,None,None,None,None,None,POINT (161430.114 -444232.136)
4,6adf6cd9b6d24ab4ee8ee220e3697a73,20206,eda9e3eb339b7f510babcd4ee0999f85,Broadway / Pacific,NaN,NaN,NaN,108.0,NaN,NaN,None,None,None,None,None,POINT (160059.661 -428400.967)


In [20]:
type(stops.rename_geometry("first_stop_geometry")[["feed_key", "stop_id", "first_stop_geometry"]]["first_stop_geometry"])

geopandas.geoseries.GeoSeries

In [21]:
# Get stop ids for the first and last trips as a DataFrame
stop_times_grouped_by_trip = stop_times.groupby(["feed_key", "trip_id"])
first_stops = stop_times.loc[
    stop_times_grouped_by_trip["stop_sequence"].idxmin()
].set_index(["feed_key", "trip_id"])
last_stops = stop_times.loc[
    stop_times_grouped_by_trip["stop_sequence"].idxmax()
].set_index(["feed_key", "trip_id"])
first_last_stops = pd.concat([
    first_stops["stop_id"].rename("first_stop_id"),
    last_stops["stop_id"].rename("last_stop_id"),
], axis=1)

In [22]:
# This isn't the most efficient, since it repeats a lot of distance calculations, but I don't know that
# it's slow enough for it to be worth adding an extra step 

# Get the geometries of the first stop
gdf_first_stops = gpd.GeoDataFrame(
    first_last_stops.reset_index().merge(
        stops.rename_geometry(
            "first_stop_geometry"
        )[["feed_key", "stop_id", "first_stop_geometry"]],
        left_on=["feed_key", "first_stop_id"],
        right_on=["feed_key", "stop_id"],
        validate="many_to_one",
    ),
    geometry="first_stop_geometry"
).sjoin_nearest( # Get the distance from the first stop to the nearest interstate
    interstates.rename_geometry("_interstate_geometry_drop")[["_interstate_geometry_drop"]],
    how="left",
    distance_col="first_stop_shortest_distance_hwy"
).drop(
    "index_right", axis=1
)
# Merge in the last stops and get their distance to the nearest intersection
gdf_first_last_stops = gdf_first_stops.merge(
    stops.rename_geometry(
        "last_stop_geometry",
    )[["feed_key", "stop_id", "last_stop_geometry"]],
    how="left",
    left_on=["feed_key", "last_stop_id"],
    right_on=["feed_key", "stop_id"],
    validate="many_to_one"
).set_geometry(
    "last_stop_geometry"
).sjoin_nearest(
    interstates.rename_geometry("_interstate_geometry_drop")[["_interstate_geometry_drop"]],
    how="left",
    distance_col="last_stop_shortest_distance_hwy"
).drop(
    "index_right", axis=1
) # i love geopandas its my favorite library
#assert not gdf_first_last_stops.index.duplicated().any()
gdf_first_last_stops["trip_distance"] = gdf_first_last_stops["first_stop_geometry"].distance(gdf_first_last_stops["last_stop_geometry"])
gdf_first_last_stops["shortest_distance_hwy"] = gdf_first_last_stops[
    ["first_stop_shortest_distance_hwy", "last_stop_shortest_distance_hwy"]
].min(axis=1)
gdf_first_last_stops.head()

,feed_key,trip_id,first_stop_id,last_stop_id,stop_id_x,first_stop_geometry,first_stop_shortest_distance_hwy,stop_id_y,last_stop_geometry,last_stop_shortest_distance_hwy,trip_distance,shortest_distance_hwy
0,008d5112a7e531d0562d26e34d77869d,1080037,8908,2425,8908,POINT (-131112.823 73193.911),2370.287926,2425,POINT (-129314.565 59399.472),2084.015768,13911.156268,2084.015768
1,008d5112a7e531d0562d26e34d77869d,1080038,8908,2425,8908,POINT (-131112.823 73193.911),2370.287926,2425,POINT (-129314.565 59399.472),2084.015768,13911.156268,2084.015768
2,008d5112a7e531d0562d26e34d77869d,1080039,8908,2425,8908,POINT (-131112.823 73193.911),2370.287926,2425,POINT (-129314.565 59399.472),2084.015768,13911.156268,2084.015768
3,008d5112a7e531d0562d26e34d77869d,1080040,8908,2425,8908,POINT (-131112.823 73193.911),2370.287926,2425,POINT (-129314.565 59399.472),2084.015768,13911.156268,2084.015768
4,008d5112a7e531d0562d26e34d77869d,1080041,8908,2425,8908,POINT (-131112.823 73193.911),2370.287926,2425,POINT (-129314.565 59399.472),2084.015768,13911.156268,2084.015768


In [23]:
# Remove duplicate entries
gdf_first_last_stops_deduplicated = gdf_first_last_stops.drop_duplicates(
    subset=["feed_key", "trip_id", "first_stop_id", "last_stop_id"]
).drop(["stop_id_x", "stop_id_y"], axis=1).set_index(["feed_key", "trip_id"])
# Get a summary of the results
trip_summary = gdf_first_last_stops_deduplicated[["trip_distance", "shortest_distance_hwy"]]
trip_summary.head()

trip_distance  shortest_distance_hwy
feed_key                         trip_id                                      
008d5112a7e531d0562d26e34d77869d 1080037   13911.156268            2084.015768
                                 1080038   13911.156268            2084.015768
                                 1080039   13911.156268            2084.015768
                                 1080040   13911.156268            2084.015768
                                 1080041   13911.156268            2084.015768

In [24]:
# Print a summary of the results
trip_summary.describe()

,trip_distance,shortest_distance_hwy
count,1.090700e+05,1.090950e+05
mean,1.303534e+04,2.646964e+04
std,2.789725e+04,2.264590e+05
min,0.000000e+00,1.156754e+00
25%,2.964591e+03,3.525033e+02
50%,8.590604e+03,7.686085e+02
75%,1.641999e+04,2.566391e+03
max,1.155340e+06,3.838715e+06
